# SMETANA and SteadyCom

In [110]:
import reframed
import pandas as pd
import numpy as np
from os import listdir
import warnings
import seaborn as sns
from collections import OrderedDict
import json

In [111]:
import sys

sys.path.append("../functions/")
from EGC import EGC_identifier

In [112]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


**Load relative abundance (E. coli)**

In [113]:
relative_abundance_ecoli = pd.read_csv("../data/Guillen2021/relative_abundance.csv",index_col=0)
relative_abundance_ecoli.index = relative_abundance_ecoli.index.map(lambda x:x if x!="yhhK" else "panZ")

**Load models**

*E. coli mutants  (top 7)*

In [114]:
RA_ecoli_mean = relative_abundance_ecoli.T.mean().sort_values(ascending=False)
top7 = RA_ecoli_mean[RA_ecoli_mean>0.01].index
top7_string = [f"mutant_{mutant}.xml" for mutant in top7]

In [115]:
models_ecoli = {}
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')
    for path in listdir("../models/Guillen2021/strains/"):
        if path not in top7_string: continue
        models_ecoli[path.split("_")[1][:-4]] = reframed.load_cbmodel("../models/Guillen2021/strains/"+path,exchange_detection="^R_EX_(?!nad_c$)(?!thf_c$).*")

*Gut microbiome members*

In [116]:
models_gut = {}
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')
    for path in listdir("../models/Venturelli2018/species_used_noO2/"):
        models_gut[path[:-4]] = reframed.load_cbmodel("../models/Venturelli2018/species_used_noO2/"+path,exchange_detection="R_EX_(?!biomass__40__e__41__)")


**Load media**

*For E. coli mutants (LB medium to see auxotrophies)*

In [128]:
media_db = pd.read_csv("https://raw.githubusercontent.com/cdanielmachado/carveme/refs/heads/master/carveme/data/input/media_db.tsv",sep="\t")
compounds = list(media_db[media_db.medium=="M9"].compound.values)
env_ecoli = reframed.Environment.from_compounds(compounds,max_uptake=50)

*For gut microbiome members*

In [129]:
media_db_gut = pd.read_csv("../data/Venturelli2018/generated/media_db_lb_extend.tsv",sep="\t")
reactions = media_db_gut[media_db_gut.medium=="LB_extend"].compound.values
#reactions = [rxn.replace("91","40").replace("93","41") for rxn in reactions]

env_gut = reframed.Environment.from_reactions(reactions,max_uptake=50)

**Load other data**

In [119]:
gene_id2name = pd.read_csv("../data/Guillen2021/generated/gene_id2name.tsv",sep="\t")

### Check community growth

**Make community model**

*For E. coli mutants*

In [120]:
comm_ecoli = reframed.Community(models=models_ecoli.values(),community_id="ecoli_mutants")

*For gut microbiome members*

In [121]:
comm_gut = reframed.Community(models=models_gut.values(),community_id="gut_microbiome")

**Test community growth**

*E. coli mutants*

In [130]:
comm_ecoli = reframed.Community(community_id="ecoli_mutants",models=models_ecoli.values(),copy_models=True)

env_ecoli.apply(comm_ecoli.merged_model, inplace=True, exclusive=True, warning=False)

biomass_rxns = comm_ecoli.merged_model.get_metabolite_producers("community_biomass")
growth = np.ones(len(biomass_rxns))*0.1
growth_upper = np.ones(len(biomass_rxns))*1000
constraints_growth_ecoli = dict(zip(biomass_rxns,list(zip(growth,growth_upper))))

In [133]:
sol = reframed.FBA(comm_ecoli.merged_model,
                   constraints=constraints_growth_ecoli
                  )
sol

Objective: 2.544000217675812
Status: Optimal

In [134]:
for rxn in biomass_rxns:
    print(f"{rxn}: {sol.values[rxn]}")

R_BIOMASS_Ec_iML1515_core_75p37M_mutant_panZ: 0.1
R_BIOMASS_Ec_iML1515_core_75p37M_mutant_bioF: 1.944000217675812
R_BIOMASS_Ec_iML1515_core_75p37M_mutant_bioC: 0.1
R_BIOMASS_Ec_iML1515_core_75p37M_mutant_icd: 0.1
R_BIOMASS_Ec_iML1515_core_75p37M_mutant_bioH: 0.1
R_BIOMASS_Ec_iML1515_core_75p37M_mutant_cysW: 0.1
R_BIOMASS_Ec_iML1515_core_75p37M_mutant_panD: 0.1


*Gut microbiome*

In [135]:
comm_gut = reframed.Community(community_id="gut_microbiome",models=models_gut.values(),copy_models=True)

env_gut.apply(comm_gut.merged_model, inplace=True, exclusive=True, warning=False)

biomass_rxns = comm_gut.merged_model.get_metabolite_producers("community_biomass")
growth = np.ones(len(biomass_rxns))*0.1
growth_upper = np.ones(len(biomass_rxns))*1000
constraints_growth_gut = dict(zip(biomass_rxns,list(zip(growth,growth_upper))))

In [136]:
sol = reframed.FBA(comm_gut.merged_model,constraints=constraints_growth_gut)

sol.values["community_growth"]

109.19383880000572

In [137]:
for rxn in biomass_rxns:
    print(f"{rxn}: {sol.values[rxn]}")

R_biomass536_M_Bacteroides_uniformis_ATCC_8492: 0.1
R_biomass345_M_Bacteroides_vulgatus_ATCC_8482: 0.1
R_biomass524_M_Collinsella_aerofaciens_ATCC_25986: 0.1
R_biomass223_M_Faecalibacterium_prausnitzii_A2_165: 0.1
R_biomass467_M_Eggerthella_lenta_DSM_2243: 0.1
R_biomass525_M_Prevotella_copri_CB7_DSM_18205: 0.1
R_biomass525_M_Clostridium_hiranonis_TO_931_DSM_13275: 0.1
R_biomass205_M_Eubacterium_rectale_ATCC_33656: 0.1
R_biomass525_M_Desulfovibrio_piger_ATCC_29098: 0.1
R_biomass536_M_Bacteroides_ovatus_ATCC_8483: 0.1
R_biomass345_M_Bacteroides_thetaiotaomicron_VPI_5482: 108.09383880000573
R_biomass525_M_Blautia_hydrogenotrophica_DSM_10507: 0.1
